# 05 -- Animation (DCC process D1, dissemination/outreach)

**SEA-FORWARD** OceanPrediction-A toolkit

Time-evolving companion to `01_seaforward_postprocess_plot.ipynb`'s static
maps: five animations over a CROCO run, all built on a single entry
point, `sftools.animation.animate(ds, var, overlay=...)`.

| Section | Animation | Call |
|---|---|---|
| 1 | Sea Surface Temperature & wind stress | `anim.animate(ds, "temp", overlay="wind")` |
| 2 | Sea Surface Height & surface currents | `anim.animate(ds, "zeta", overlay="uv")` |
| 3 | Current speed & quivers | `anim.animate(ds, "speed", overlay="uv")` |
| 4 | Zonal current (u) | `anim.animate(ds, "u", overlay="uv")` |
| 5 | Meridional current (v) | `anim.animate(ds, "v", overlay="uv")` |

**Note on the module name.** This notebook imports `sftools.animation`
(not `sftools.animate`, used by an earlier revision of this notebook) --
if your checkout still only has `sftools/animate.py`, either update to the
version that provides `sftools/animation.py`, or adjust the import below.

*Language note (FR-09):* markdown and docstrings are in English; French
translation is coordinated separately with the documentation team.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, "..")   # repo root, so `import sftools...` resolves

%matplotlib inline
import importlib
import os, re
import sftools
import sftools.postprocess as pp
import sftools.plotting as pl
import sftools.animation as anim
importlib.reload(pp)
importlib.reload(pl)
importlib.reload(anim)


## 0. Find and select a cycle

Same discovery pattern as `02_validation.ipynb` Section 1 (and
`01_seaforward_postprocess_plot.ipynb` Section 0): every run lives in a
cycle directory named `YYYYMMDD`, sibling to every other cycle under
`<MAIN_DIR>/<CONFIG>/`. `RUN_TYPE` distinguishes a forecast cycle
(`fcst/CROCO_FILES/croco_his.nc`) from a hindcast cycle
(`hcast/CROCO_FILES/croco_his.nc`) -- this notebook's original example
pointed at a hindcast run, so that's the default here. Every available
cycle under `MAIN_DIR/CONFIG` is listed below; pick one with `CYCLE` (or
the `SEAFORWARD_CYCLE` environment variable).


In [ ]:
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
RUN_TYPE = os.environ.get("SEAFORWARD_RUN_TYPE", "fcst")   # "fcst" or "hcast"
MAIN_DIR = os.path.expanduser(
    os.environ.get("SEAFORWARD_MAIN_DIR",
                   f"~/seaforward/{'forecast' if RUN_TYPE == 'fcst' else 'hindcast'}/model-runs"))

def _his_path(main_dir, config, cycle, run_type):
    return os.path.join(main_dir, config, cycle, run_type, "CROCO_FILES", "croco_his.nc")

def _list_cycles(main_dir, config, run_type):
    base = os.path.join(main_dir, config)
    if not os.path.isdir(base):
        return []
    return sorted(name for name in os.listdir(base)
                 if re.match(r"^\d{8}$", name)
                 and os.path.exists(_his_path(main_dir, config, name, run_type)))

AVAILABLE_CYCLES = _list_cycles(MAIN_DIR, CONFIG, RUN_TYPE)
print(f"{RUN_TYPE} cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")


In [ ]:

# >>> SET THIS to the cycle you want to open, e.g. "20251225_plain" <<<
CYCLE = os.environ.get("SEAFORWARD_CYCLE", "20260729_plain")

# hindcast runs from the GLORYS reanalysis (CF time origin year 1993);
# forecast runs from the Copernicus Marine Forecast / Mercator anfc (2000)
YORIG = 1993 if RUN_TYPE == "hcast" else 2000

H = _his_path(MAIN_DIR, CONFIG, CYCLE, RUN_TYPE)
if not os.path.exists(H):
    raise FileNotFoundError(
        f"No CROCO history file for cycle {CYCLE!r} at {H}.\n"
        f"Available {RUN_TYPE} cycles under {os.path.join(MAIN_DIR, CONFIG)}: "
        f"{AVAILABLE_CYCLES if AVAILABLE_CYCLES else '(none found)'}")

print(f"Opening {RUN_TYPE} cycle {CYCLE}")
print(f"  CROCO history: {H}")


## 1. Load CROCO Dataset

In [ ]:
ds = pp.open_history(H, Yorig=YORIG)
print(f"Dataset loaded: {len(ds['time'])} time steps.")

## context about anim.animate()

`anim.animate(ds, var, depth_m=None, overlay=None, uv_depth=None, isobaths=None, tindex_range=None, skip=4,
            scale=None, interval=300, figsize=(8, 7), cmap=None, vmin=None, vmax=None, out=None, fps=4, dpi=110)`

## Animate a 2D field through the run.

## Parameters
- **ds : xarray.Dataset**
    CROCO output opened with ``pp.open_history()`` or ``pp.open_run()``.
- **var : str**
    ``'temp'``, ``'salt'``, ``'zeta'``, ``'speed'``, ``'u'``, ``'v'``, ...
- **overlay : str or None**
    ``'wind'`` / ``'sustr'`` — surface wind-stress vectors, taken from the
    model's own ``sustr``/``svstr`` (N/m², not 10 m wind speed).
    ``'uv'`` / ``'current'`` — surface current vectors.
    ``None`` — no overlay.
- **isobaths : list of float, optional**
    Depths (m) to contour, e.g. ``[200, 1000]``.
- **tindex_range : (start, end), optional**
    Range of time indices. Default: every record.
- **skip : int**
    Vector subsampling stride (default 4).
- **scale : float, optional**
    Quiver scale. Smaller values give longer arrows.
- **interval : int**
    Frame delay in milliseconds, for the inline widget.
figsize : tuple
    Figure size in inches.
- **cmap, vmin, vmax : optional**
    Colour overrides. Limits are resolved **once** across the whole series,
    so the scale does not flicker between frames.
- **out : str, optional**
    Write to this path instead of returning a widget. ``.gif`` uses the
    pillow writer, anything else uses ffmpeg.
fps : int
    Frames per second when writing a file.
- **dpi : int**
    Resolution when writing a file.


# Some parameters

In [ ]:
depth = None
dpi   = 300

## Define Directory to save animations

In [ ]:
out_dir =  os.path.join(MAIN_DIR, CONFIG,  f"animation_{CYCLE}")
os.makedirs(out_dir, exist_ok=True)
print(out_dir)

## 2. Animation: Sea Surface Temperature (SST) & Wind Stress

Shaded Sea Surface Temperature (`temp`) overlaid with surface wind stress
vectors -- the time-evolving counterpart to
`01_seaforward_postprocess_plot.ipynb`'s static SST + wind stress map: the
cold tongue, its front, and the filaments carrying water offshore,
animated frame by frame so you can see the front respond to the wind
rather than reading it off a single instant.


In [ ]:
anim.animate(ds, "temp",  depth_m= depth, overlay="wind",  vmin=22, vmax=29)

#### to save animation
# z=[0 if depth is None else depth]
# anim.animate(ds, "temp",  depth_m=depth, overlay="wind", out=f"{out_dir}/temp_animation_{z[0]}m_cycle-{CYCLE}.gif")

## 3. Animation: Sea Surface Height (SSH) & Surface Currents

Shaded Sea Surface Elevation (`zeta`) overlaid with surface current
velocity vectors ($u$, $v$). SSH is the pressure field the geostrophic
flow follows, so watching the highs/lows migrate alongside the current
vectors is the quickest way to see eddies form, drift and decay over the
run.


In [ ]:
anim.animate(ds, "zeta", overlay="uv", cmap='Spectral_r')

#### to save animation
# anim.animate(ds, "zeta", overlay="uv", out=f"{out_dir}/SSH_animation_cycle-{CYCLE}.gif")

## 4. Animation: Current Speed & Quivers

Current speed magnitude with directional quivers -- draws a quiver of
current vectors over a speed-shaded background, highlighting where the
flow is fastest (jets, eddy cores) as it evolves in time.


In [ ]:
anim.animate(ds, "speed", depth_m=depth, overlay="uv")

#### to save animation
# z=[0 if depth is None else depth]
# anim.animate(ds, "speed",  depth_m=depth, overlay="uv", out=f"{out_dir}/speed_animation_{z[0]}m_cycle-{CYCLE}.gif")

## 5. Animation: Zonal Current (u)

Eastward velocity component ($u$) with directional quivers, isolating the
zonal (east-west) part of the flow.


In [ ]:
anim.animate(ds, "u", overlay="uv")

#### to save animation
# z=[0 if depth is None else depth]
# anim.animate(ds, "u",  depth_m=depth, overlay="uv", out=f"{out_dir}/u_animation_{z[0]}m_cycle-{CYCLE}.gif")

## 6. Animation: Meridional Current (v)

Northward velocity component ($v$) with directional quivers, isolating
the meridional (north-south) part of the flow -- compare against Section
5 to see which component dominates the circulation at a given time/place.


In [ ]:
anim.animate(ds, "v", overlay="uv")

#### to save animation
# z=[0 if depth is None else depth]
# cmap='Spectral_ranim.animate(ds, "v",  depth_m=depth, overlay="uv", out=f"{out_dir}/v_animation_{z[0]}m_cycle-{CYCLE}.gif")

---
## Notes

- **DCC linkage (FR-12):** animation is a D1 (downstream/dissemination)
  product -- it consumes validated C1 output (ideally already checked by
  `02_validation.ipynb`, DCC process V1) and produces outreach/diagnostic
  material, not a new analysis result in its own right. See
  [sea-forward.readthedocs.io](https://sea-forward.readthedocs.io/en/latest/)
  for the full Upstream -> C1 -> V1 -> D1 value chain this notebook sits at
  the end of.
- **Runtime:** animating a full-size regional run with many time steps
  will take noticeably longer than a single static plot from
  `01_postprocessing.ipynb` -- each frame is a fresh rende.
